In [20]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from pathlib import Path
from PIL import Image
import glob
import os
import sys
import pandas as pd
import numpy as np 

In [21]:
KAGGLE_DATA_ROOT = Path("../data/raw/chest_xray")
PROCESSED_DIR = Path("data/processed")

In [29]:
#Image and batch settings
IMG_SIZE = 224
BATCH_SIZE = 32
RANDOM_SEED = 42 # for reproducible splits

In [30]:
def gather_all_data(root_dir):
    """
    Walks through kaggle's train/test/val folders to collect all image paths and lables.
    """
    print(f"Scanning for data in: {root_dir}")
    image_paths = []
    labels = []

    # Kaggle dataset has 'train', 'test', and 'val' subfolders
    subsets = ['train', 'test', 'val']
    
    for subset in subsets:
        subset_path = root_dir / subset
        if not subset_path.exists():
            print(f"Warning: Directory not found, skipping: {subset_path}")
            continue
            
        # Get Normal cases (Label: 0)
        # We use glob to find all .jpeg files
        normal_paths = list(subset_path.glob('NORMAL/*.jpeg'))
        image_paths.extend(normal_paths)
        labels.extend([0] * len(normal_paths)) # 0 for Normal
        
        # Get Pneumonia cases (Label: 1)
        pneumonia_paths = list(subset_path.glob('PNEUMONIA/*.jpeg'))
        image_paths.extend(pneumonia_paths)
        labels.extend([1] * len(pneumonia_paths)) # 1 for Pneumonia
        
    if not image_paths:
        print(f"Error: No images found in {root_dir}.")
        print("Please check that KAGGLE_DATA_ROOT is set correctly.")
        sys.exit(1)
        
    print(f"Total images found: {len(image_paths)}")
    return image_paths, labels

In [31]:
# --- 3. CREATE 80/20 STRATIFIED SPLIT ---

def create_and_save_splits(paths, labels, output_dir):
    """
    Splits data into 80/20 train/val sets and saves them to CSVs
    in the data/processed directory.
    """
    
    # Split the data: 80% Train, 20% Validation
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        paths, 
        labels, 
        test_size=0.2, 
        random_state=RANDOM_SEED, 
        stratify=labels  # Crucial: Maintains class balance
    )

    print(f"Training set size: {len(train_paths)}")
    print(f"Validation set size: {len(val_paths)}")

    # Create DataFrames
    # We convert paths to strings for easier saving in CSV
    train_df = pd.DataFrame({
        'image_path': [str(p) for p in train_paths],
        'label': train_labels
    })
    
    val_df = pd.DataFrame({
        'image_path': [str(p) for p in val_paths],
        'label': val_labels
    })

    # Ensure the processed directory exists
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save the splits as CSV files
    train_csv_path = output_dir / "train_split.csv"
    val_csv_path = output_dir / "val_split.csv"
    
    train_df.to_csv(train_csv_path, index=False)
    val_df.to_csv(val_csv_path, index=False)
    
    print(f"Training split saved to: {train_csv_path}")
    print(f"Validation split saved to: {val_csv_path}")
    
    return train_df, val_df

In [32]:
# --- 4. DATA LOADING FUNCTION (PYTORCH DATASET CLASS) ---
# This class will be used in your *training* script, 
# but we define it here for completeness.

class PneumoniaDataset(Dataset):
    """
    PyTorch Dataset class for loading data from our processed CSV splits.
    """
    def __init__(self, csv_file, transform=None):
        """
        Args:
            csv_file (str or Path): Path to the CSV file (train_split.csv or val_split.csv).
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.df = pd.read_csv(csv_file)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # Get path and label from the DataFrame
        img_path = self.df.iloc[idx, 0]
        label = self.df.iloc[idx, 1]
        
        # 1. Open Image
        # We convert to 'RGB' because X-rays are grayscale (1 channel) 
        # but most pre-trained CNNs (like ResNet) expect 3 channels.
        img = Image.open(img_path).convert("RGB")
        
        # 2. Apply Transformations
        # This will handle resizing and normalization (via ToTensor())
        if self.transform:
            img = self.transform(img)
            
        # 3. Return image tensor and label tensor
        # Label is cast to float for compatibility with BCELoss
        return img, torch.tensor(label, dtype=torch.float32)

In [33]:
# --- 5. VERIFY SPLIT AND DEMONSTRATE DATA LOADING ---

def verify_splits(train_df, val_df):
    """
    Prints the class balance for both training and validation sets.
    """
    print("\n--- (5) VERIFICATION OF SPLITS ---")
    
    print("\nTraining Set Class Distribution:")
    train_counts = train_df['label'].value_counts(normalize=True)
    print(f"Normal (0): {train_counts[0]:.2%}")
    print(f"Pneumonia (1): {train_counts[1]:.2%}")
    
    print("\nValidation Set Class Distribution:")
    val_counts = val_df['label'].value_counts(normalize=True)
    print(f"Normal (0): {val_counts[0]:.2%}")
    print(f"Pneumonia (1): {val_counts[1]:.2%}")
    
    if abs(train_counts[0] - val_counts[0]) < 0.01:
        print("\nVerification successful: Stratification maintained class balance.")
    else:
        print("\nVerification warning: Class balance differs significantly between sets.")

def demonstrate_dataloader():
    """
    Creates and tests the DataLoaders to show they work.
    """
    print("\n--- DEMONSTRATING DATALOADER ---")
    
    # Define the transformation pipeline
    # This implements: Resize to 224x224, Normalize (divide by 255)
    data_transforms = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)), # Resize to 224x224
        transforms.ToTensor(),                   # Converts PIL Image [0-255] to Tensor [0-1]
                                                 # This automatically handles normalization!
    ])
    
    # Instantiate the datasets by loading the CSVs we created
    train_dataset = PneumoniaDataset(
        csv_file=PROCESSED_DIR / "train_split.csv", 
        transform=data_transforms
    )
    
    # Create a DataLoader
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True
    )
    
    # Get one batch of data
    try:
        images, labels = next(iter(train_loader))
        
        print(f"Successfully loaded one batch.")
        print(f"Images batch shape: {images.shape}")  # [Batch, Channels, Height, Width]
        print(f"Labels batch shape: {labels.shape}")
        print(f"Image tensor min value: {images.min()}")
        print(f"Image tensor max value: {images.max()}")
        print(f"Example labels: {labels[:5]}")
        print("\nData preprocessing pipeline is ready!")
        
    except Exception as e:
        print(f"\nError loading data: {e}")
        print("Please check file paths and image integrity.")

In [34]:
# --- MAIN EXECUTION ---
if __name__ == "__main__":
    print("--- STEP 3: DATA PREPROCESSING AND SPLIT CREATION ---")
    
    # Step 2:
    all_paths, all_labels = gather_all_data(KAGGLE_DATA_ROOT)
    
    # Step 3:
    train_df, val_df = create_and_save_splits(all_paths, all_labels, PROCESSED_DIR)
    
    # Step 5 (Verification):
    verify_splits(train_df, val_df)
    
    # Bonus: Demonstrate the loader works
    demonstrate_dataloader()

--- STEP 3: DATA PREPROCESSING AND SPLIT CREATION ---
Scanning for data in: ..\data\raw\chest_xray
Total images found: 5856
Training set size: 4684
Validation set size: 1172
Training split saved to: data\processed\train_split.csv
Validation split saved to: data\processed\val_split.csv

--- (5) VERIFICATION OF SPLITS ---

Training Set Class Distribution:
Normal (0): 27.03%
Pneumonia (1): 72.97%

Validation Set Class Distribution:
Normal (0): 27.05%
Pneumonia (1): 72.95%

Verification successful: Stratification maintained class balance.

--- DEMONSTRATING DATALOADER ---
Successfully loaded one batch.
Images batch shape: torch.Size([32, 3, 224, 224])
Labels batch shape: torch.Size([32])
Image tensor min value: 0.0
Image tensor max value: 1.0
Example labels: tensor([0., 1., 1., 1., 1.])

Data preprocessing pipeline is ready!
